# Greenhouse Slug Discovery

This notebook tests candidate Greenhouse board slugs against:

`https://boards-api.greenhouse.io/v1/boards/{slug}/jobs`

A slug is considered **working** when the API responds `200 OK`.

## prep data:

```bash
sed -E '/Logo$/d;/^[0-9]+(-[0-9]+)?\+?$/d;/^$/d' companies.txt > companies_clean.txt
```
```bash
awk 'BEGIN{print "company"} {gsub(/"/,"\"\""); print "\"" $0 "\""}' companies_clean.txt > companies.csv
```

In [6]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import pandas as pd


In [7]:
# --- Configuration ---
BASE_URL = "https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 12
SLEEP_JITTER_SEC = (0.0, 0.15)  # polite jitter between calls in each worker

# Seed list from the app's current curated companies.
seed_slugs = [
    "airbnb", "amplitude", "anthropic", "asana", "benchling", "brex", "canva", "cloudflare",
    "coinbase", "datadog", "databricks", "discord", "doordash", "dropbox", "duolingo",
    "figma", "fivetran", "gitlab", "gusto", "hashicorp", "hubspot", "intercom", "lattice",
    "launchdarkly", "linear", "loom", "lyft", "mercury", "miro", "mixpanel", "mongodb",
    "notion", "pagerduty", "palantir", "pinterest", "plaid", "ramp", "reddit", "retool",
    "rippling", "robinhood", "scale", "segment", "snowflake", "stripe", "twilio", "verkada",
    "zendesk", "zoom"
]

# Add your own guesses here (lowercase slug format):
extra_candidates = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/companies.csv")["company"].dropna().tolist()

candidates = sorted(set(seed_slugs + extra_candidates))
print(f"Total candidate slugs: {len(candidates)}")

Total candidate slugs: 3026


In [8]:
def probe_slug(slug: str) -> dict:
    url = BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        # jitter to be polite when running high concurrency
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(r.json().get("jobs", [])) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_slugs(slugs: Iterable[str]) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [9]:
results = scan_slugs(candidates)
results.head(200)

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,2U,200,True,30.0,https://boards-api.greenhouse.io/v1/boards/2U/...,724.7
1,6sense,200,True,32.0,https://boards-api.greenhouse.io/v1/boards/6se...,118.3
2,ASG,200,True,3.0,https://boards-api.greenhouse.io/v1/boards/ASG...,190.2
3,Accela,200,True,12.0,https://boards-api.greenhouse.io/v1/boards/Acc...,203.7
4,Accordion,200,True,47.0,https://boards-api.greenhouse.io/v1/boards/Acc...,180.2
...,...,...,...,...,...,...
195,Marqeta,200,True,17.0,https://boards-api.greenhouse.io/v1/boards/Mar...,108.2
196,MasterControl,200,True,9.0,https://boards-api.greenhouse.io/v1/boards/Mas...,97.5
197,McAfee,200,True,3.0,https://boards-api.greenhouse.io/v1/boards/McA...,130.6
198,Melio,200,True,28.0,https://boards-api.greenhouse.io/v1/boards/Mel...,98.9


In [10]:
results["url"].tail(200)

2826    https://boards-api.greenhouse.io/v1/boards/Vox...
2827    https://boards-api.greenhouse.io/v1/boards/Vox...
2828    https://boards-api.greenhouse.io/v1/boards/Voy...
2829    https://boards-api.greenhouse.io/v1/boards/Vpa...
2830    https://boards-api.greenhouse.io/v1/boards/Vub...
                              ...                        
3021    https://boards-api.greenhouse.io/v1/boards/sym...
3022    https://boards-api.greenhouse.io/v1/boards/tes...
3023    https://boards-api.greenhouse.io/v1/boards/zen...
3024    https://boards-api.greenhouse.io/v1/boards/zip...
3025    https://boards-api.greenhouse.io/v1/boards/zoo...
Name: url, Length: 200, dtype: object

In [11]:
summary = (
    results.assign(status_group=results["status_code"].fillna(-1))
    .groupby("status_group", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("status_group")
)
summary

,status_group,count
0,200,382
1,404,2644


In [ ]:
working = results[results["ok"]].copy()
print(f"Working slugs: {len(working)}")
working[["slug", "jobs_count", "status_code", "elapsed_ms"]].head()

Working slugs: 382


50

In [13]:
# Save only the working slugs — one per line, lowercase, no index, no header.
# Ready to upload with: node scripts/import-slugs.mjs notebooks/greenhouse_slug_scan_results.csv
output_csv = "greenhouse_slug_scan_results.csv"
gh_working = results[results["ok"]].copy()
gh_slugs = (
    gh_working["slug"]
    .str.lower()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
gh_slugs.to_csv(output_csv, index=False, header=False)
print(f"Saved {len(gh_slugs)} slugs to {output_csv}")

Saved 357 slugs to greenhouse_slug_scan_results.csv


---

## Ashby API Check

Testing the same candidate slugs against the Ashby public job board API:

`https://api.ashbyhq.com/posting-api/job-board/{slug}`

- `200` means the company has an Ashby job board at that slug.
- `404` / other means no board found.

> Slugs are case-sensitive in Ashby — this first pass uses lowercase (same as Greenhouse slugs).

In [14]:
ASHBY_BASE_URL = "https://api.ashbyhq.com/posting-api/job-board/{slug}"

def probe_ashby_slug(slug: str) -> dict:
    url = ASHBY_BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        jobs = []
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobs", [])

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(jobs) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_ashby_slugs(slugs) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_ashby_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [15]:
ashby_results = scan_ashby_slugs(candidates)

ashby_working = ashby_results[ashby_results["ok"]].copy()
print(f"Working Ashby slugs: {len(ashby_working)} / {len(candidates)}")
ashby_results

Working Ashby slugs: 146 / 3026


,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Abridge,200,True,64.0,https://api.ashbyhq.com/posting-api/job-board/...,1880.2
1,Acorns,200,True,12.0,https://api.ashbyhq.com/posting-api/job-board/...,411.1
2,Affinity,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,607.8
3,Airtable,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,388.1
4,Andela,200,True,16.0,https://api.ashbyhq.com/posting-api/job-board/...,563.1
...,...,...,...,...,...,...
3021,twilio,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,184.6
3022,xAI,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/xAI,179.3
3023,zendesk,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,194.6
3024,zip2,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,199.4


In [16]:
ashby_working.head()

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Abridge,200,True,64.0,https://api.ashbyhq.com/posting-api/job-board/...,1880.2
1,Acorns,200,True,12.0,https://api.ashbyhq.com/posting-api/job-board/...,411.1
2,Affinity,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,607.8
3,Airtable,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,388.1
4,Andela,200,True,16.0,https://api.ashbyhq.com/posting-api/job-board/...,563.1


In [17]:
ashby_clean = ashby_working["slug"]
ashby_clean.to_csv("ashby_slug_scan_results.csv")

In [18]:
csv_greenhouse = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/greenhouse_slug_scan_results.csv")
csv_ashby = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/ashby_slug_scan_results.csv")

In [19]:
csv_greenhouse.head()


,2u
0,6sense
1,accela
2,accordion
3,acquia
4,affinitiv


In [20]:
csv_ashby.head()

,Unnamed: 0,slug
0,0,Abridge
1,1,Acorns
2,2,Affinity
3,3,Airtable
4,4,Andela


In [21]:
# Save only the working Ashby slugs — one per line, lowercase, no index, no header.
# Ready to upload with: node scripts/import-slugs.mjs --table ashby_slugs notebooks/ashby_slug_scan_results.csv
ashby_output_csv = "ashby_slug_scan_results.csv"
ashby_slugs = (
     csv_ashby["slug"]
    .str.lower()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
ashby_slugs.to_csv(ashby_output_csv, index=False, header=False)
print(f"Saved {len(ashby_slugs)} slugs to {ashby_output_csv}")

Saved 139 slugs to ashby_slug_scan_results.csv


---

## Upload Slugs to the Database

Run these commands from the **root of the project** after running the save cells above.

### Greenhouse

```bash
node scripts/import-slugs.mjs notebooks/greenhouse_slug_scan_results.csv
```

### Ashby

```bash
node scripts/import-slugs.mjs --table ashby_slugs notebooks/ashby_slug_scan_results.csv
```

> Re-running either command is safe — existing slugs are skipped (`ON CONFLICT DO NOTHING`).  
> Make sure `.env.local` contains `NEXT_PUBLIC_SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`.